# Genotype–Phenotype Heterogeneity Dataset Exploration with `mlcroissant`
This notebook provides a walkthrough for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

This dataset contains structured tabulations of clinical features, hormone levels, adrenal imaging, and genetic variants for three female patients diagnosed with non-classical 21-hydroxylase deficiency-associated infertility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The Croissant schema URL defines the dataset structure and record sets.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print("Dataset Title: ", metadata['name'])
print("Description: ", metadata['description'])
print("Published: ", metadata['datePublished'])
print("Authors (IDs): ", metadata['author'])
print("Record Sets: ", metadata['recordSet'])

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

To get a sense of the dataset structure, enumerate the record sets. For each record set, print details and field IDs. **All references will use the `@id`.**

In [ ]:
# Get all record sets defined by `@id`
record_sets = dataset.record_sets()
print(f"Number of record sets: {len(record_sets)}")
for rset in record_sets:
    print(f"RecordSet @id: {rset['@id']}")
    print(f"  Name: {rset.get('name', 'No name')}")
    if 'field' in rset:
        print("  Fields:")
        for field in rset['field']:
            print(f"    - Field @id: {field['@id']} (Name: {field.get('name', '')})")
    else:
        print("  No fields.")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s found in the previous step.

Each record set can be loaded individually. For illustration, we load all available record sets, referencing them by their `@id`.

In [ ]:
# Extract data from each record set (referenced by @id)
record_set_ids = [rset['@id'] for rset in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"RecordSet @id: {record_set_id}, columns: {df.columns.tolist()}")
    else:
        print(f"RecordSet @id: {record_set_id} contains no records.")

# For demonstration, choose the first available record set with records
if len(dataframes) > 0:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"\nPreview of DataFrame for RecordSet @id: {selected_record_set_id}")
    display(dataframes[selected_record_set_id].head())
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For this example, choose a numeric field (referenced by its `@id`) from the selected record set, filter by threshold, normalize, and optionally group.

In [ ]:
import numpy as np

# EDA for the selected record set
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    # Find numeric columns
    numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64]]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Analyzing numeric field: {numeric_field_id}")
        # Example threshold
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        
        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
        
        # Try grouping on another field in df
        possible_group_fields = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
        if possible_group_fields:
            group_field_id = possible_group_fields[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric fields found in selected record set.")
else:
    print("No record sets with data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and numeric_fields:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists, show boxplot
    if possible_group_fields:
        plt.figure(figsize=(7,5))
        sns.boxplot(x=df[possible_group_fields[0]], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {possible_group_fields[0]}")
        plt.xlabel(possible_group_fields[0])
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated:
- Loading FAIR^2 dataset metadata and tabular records using `mlcroissant`.
- Exploring record sets, fields, and extracting tables by `@id`.
- Basic filtering, normalization, and grouping of numeric fields.
- Visualization of distributions and relationships in clinical/genetic data.

The dataset is small and case-based, designed for educational and research use. Further exploration could combine phenotypic and genetic tabulations, or link field values to domain ontologies for deeper analysis.